<a href="https://colab.research.google.com/github/repulsivityy/learning-LLMs/blob/main/notebooks/00_mechanics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Initial Setup

Set ups Module 0 - Mechanics

In [1]:
import os

REPO_URL = "https://github.com/repulsivityy/learning-LLMs.git"
REPO_DIR = "/content/learning-LLMs"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

Cloning into '/content/learning-LLMs'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 38 (delta 11), reused 28 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 13.82 KiB | 4.61 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/learning-LLMs


In [2]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{token}@github.com/repulsivityy/learning-LLMs.git

In [3]:
!pip install -q torch numpy
!mkdir -p notebooks src

In [5]:
%%writefile src/tokenizer.py
from collections import defaultdict


def get_pair_counts(word_freqs):
    pair_counts = defaultdict(int)
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_counts[pair] += freq
    return pair_counts


def merge_pair(pair, word_freqs):
    new_word_freqs = {}
    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_freqs[tuple(new_word)] = freq
    return new_word_freqs


def train_bpe(word_freqs, num_merges):
    word_freqs = dict(word_freqs)
    merges = []
    for step in range(num_merges):
        pair_counts = get_pair_counts(word_freqs)
        if not pair_counts:
            break
        best_pair = max(pair_counts, key=pair_counts.get)
        word_freqs = merge_pair(best_pair, word_freqs)
        merges.append(best_pair)
        print(f"Merge {step + 1}: {best_pair}  (count={pair_counts[best_pair]})")
    return word_freqs, merges

Writing src/tokenizer.py


In [7]:
## Module 0 — Tokenization: Byte-Pair Encoding (BPE)

**Why it matters:** a language model never sees text — only a sequence of integers.
Tokenization is the text ↔ integer conversion. BPE is the algorithm nearly every
modern LLM uses to decide what those "chunks" should be.

**ELI5:** imagine a box of individual letter tiles. Every time two tiles keep
turning up next to each other ("t" + "h" in "the", "this", "that"), you glue
them into one bigger tile. Do that thousands of times and your box ends up
with tiles for whole common words, plus loose letters for anything unusual —
so you're never stuck on a word you've never seen.

[main 84a2995] Add BPE tokenizer core functions
 2 files changed, 26 insertions(+)
 rename 00_mechanics.ipynb => notebooks/00_mechanics.ipynb (100%)
 create mode 100644 src/tokenizer.py
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 697 bytes | 697.00 KiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/repulsivityy/learning-LLMs.git
   ff0afba..84a2995  main -> main


In [ ]:
import sys
sys.path.append('/content/learning-LLMs/src')

from tokenizer import get_pair_counts, merge_pair, train_bpe

### Toy corpus

The classic example from the original BPE paper (Sennrich et al., 2016) —
useful because we know exactly what the "correct" output should be, so we
can check our implementation against it.

Each word is a tuple of characters plus an end-of-word marker `</w>`, mapped
to how many times it appears in the corpus.

In [ ]:
word_freqs = {
    ('l', 'o', 'w', '</w>'): 5,
    ('l', 'o', 'w', 'e', 'r', '</w>'): 2,
    ('n', 'e', 'w', 'e', 's', 't', '</w>'): 6,
    ('w', 'i', 'd', 'e', 's', 't', '</w>'): 3,
}
word_freqs

In [ ]:
pair_counts = get_pair_counts(word_freqs)
sorted(pair_counts.items(), key=lambda x: -x[1])[:5]

### Step 2 — `merge_pair`

Fuses the winning pair everywhere it occurs.

**ELI5:** walk through each word letter by letter; every time you spot the
exact pair you're gluing, weld those two into one tile and hop over both.

In [ ]:
merged = merge_pair(('e', 's'), word_freqs)
merged

### Step 3 — `train_bpe`: the full training loop

Repeats "find the most frequent pair → merge it" for a fixed number of
steps. The **ordered list of merges is the entire trained tokenizer** — to
tokenize new text later, you replay these same merges in this same order.

In [ ]:
final_word_freqs, merges = train_bpe(word_freqs, num_merges=8)
merges
final_word_freqs

### Save your progress

Colab's runtime is ephemeral — anything not pushed disappears when it resets.

In [ ]:
!git add -A
!git commit -m "Module 0: add train_bpe and tokenizer walkthrough"
!git push